In [ ]:
import uproot
import awkward as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

file_name = '../../MCEventInfoForBinningOptimisation.root'

with uproot.open(file_name) as f:
    # Load some data from the file
    df_events = f['MCEvent'].arrays([
                                    'TrueENu',
                                    'TrueCZ',
                                    'TrueEHad',
                                    'TrueELep',
                                    'RecoENu',
                                    'RecoEHad',
                                    'RecoELep',
                                    'RecoCZ',
                                    'Sample',
                                    'EventWeight'
                                    ],
                                     library='pd')

df_events['RecoY'] = df_events['RecoEHad'] / df_events['RecoENu']

samples = {
            'FC_numu': 0,
            'FC_nue': 1,
            'PC_numu': 2,
            'PC_nue': 3,
            }

display(df_events)

In [ ]:
def create_energy_bins(cp_step, m32_step, bsm_step):

    delta_cp_bins = np.arange(0., 1., cp_step)
    mass_ordering_bins = np.arange(1, 10, m32_step)
    bsm_bins = np.arange(10, 100+bsm_step, bsm_step)

    E_nu_bins = np.concatenate([delta_cp_bins,mass_ordering_bins,bsm_bins])
    return E_nu_bins

E_nu_bins = create_energy_bins(0.2, 0.05, 0.5)
#print(f"{len(E_nu_bins)}: {E_nu_bins}")

In [ ]:
import warnings
from enum import Enum

# how each axis is optimised
class OptimiseMode(Enum):
    joint = 'joint'   # all outer bins at once
    sequential = 'sequential'   # one outer bin at a time, in OptimiseOrder

# order in which the outer bins are optimised in OptimiseMode.sequential
class OptimiseOrder(Enum):
    lowest = 'lowest' # outer bin with the lowest counts first
    index = 'index'   # bin-index order

class SampleBinOptimiser:

    default_optimise_mode = OptimiseMode.joint
    default_optimise_order = OptimiseOrder.lowest

    def __init__(self, name, df, initial_bins, axes_factors={}, min_mc=100, min_data=3., exposure=400., nominal_exposure=400., optimise_mode=None, optimise_order=None):
        
        self.name=name
    
        self.df = df.reset_index(drop=True)

        self.initial_bins=initial_bins

        self.axes=list(self.initial_bins.keys())

        self.optimised_bins=dict.fromkeys(self.initial_bins.keys())
        self.axes_factors=dict.fromkeys(self.initial_bins.keys())

        for axis, edges in self.initial_bins.items():
            # add overflow bin to reproduce numpy histogram behaviour
            # TODO check if inf is already present
            self.optimised_bins[axis] = np.concatenate([edges, [float('inf')]])
            self.df[axis + 'binIndex'] = np.digitize(self.df[axis].to_numpy(),
                                                    self.optimised_bins[axis], right=False)
            axis_factor=1.
            if axis in axes_factors:
                axis_factor=axes_factors[axis]
            self.axes_factors[axis]=axis_factor

        # constants for algo.
        self.nominal_exposure=nominal_exposure
        self.exposure=exposure
        self.exposure_scaling=exposure/nominal_exposure

        self.min_mc=min_mc
        self.min_data=min_data

        # None = use the class default, read here so changing the class default takes effect.
        # Enum members or their values are accepted; going through .value also accepts a
        # member made before this cell was re-run (it belongs to the old enum class)
        def as_enum(enum_type, value, default):
            return default if value is None else enum_type(getattr(value, 'value', value))

        # how each axis is optimised
        self.optimise_mode=as_enum(OptimiseMode, optimise_mode, self.default_optimise_mode)

        # order in which the outer bins are optimised, only used in OptimiseMode.sequential
        self.optimise_order=as_enum(OptimiseOrder, optimise_order, self.default_optimise_order)
        if self.optimise_mode is OptimiseMode.joint and optimise_order is not None:
            warnings.warn(f"optimise_order={self.optimise_order} is ignored in {self.optimise_mode}",
                          stacklevel=2)

    ###
    def optimise_bins(self):

        for i, axis in enumerate(self.axes):

            bins_array=list(self.optimised_bins.values())
            target_bins=bins_array[i]

            #print(f"optimising {axis} {target_bins}")

            outer_axes=self.axes[:i+1]
            outer_axes.pop()

            outer_bins=bins_array[:i+1]
            outer_bins.pop()

            outer_n_bins = [len(x)-2 for x in outer_bins]

            #print(f"outer {outer_axes} {outer_n_bins} {outer_bins}")

            if self.optimise_mode is OptimiseMode.joint:
                self._optimise_axis_joint(outer_n_bins, outer_axes, axis)
            else:
                self._optimise_bins_loop(outer_n_bins, outer_axes, axis)
        return self.optimised_bins
    ###

    ###
    def _build_outer_bin_sel_idx(self, outer_bins, outer_axes):
        # yield every outer bin with the events in it, in bin-index order
        # relies on the outer axes' binIndex columns not changing while an inner axis is optimised

        for indices in np.ndindex(*outer_bins):
            bin_mask = np.ones(len(self.df), dtype=bool)
            bin_indices = [idx+1 for idx in indices]
            for o_a, bin_idx in zip(outer_axes, bin_indices):
                bin_mask &= self.df[o_a+'binIndex'].to_numpy() == bin_idx
            yield bin_indices, np.flatnonzero(bin_mask)
    ###

    ###
    def _optimise_bins_loop(self, outer_bins, outer_axes, axis):
        outer_bins_sel_idx = self._build_outer_bin_sel_idx(outer_bins, outer_axes)

        if self.optimise_order is OptimiseOrder.lowest:
            # optimise the outer bin with the lowest counts first, so its merges are applied to the
            # shared edges before the better-populated outer bins are checked.
            outer_bins_sel_idx = sorted(outer_bins_sel_idx, key=lambda outer_bin: self._compute_sel_idx_axis_score(axis, outer_bin[1]))

        for bin_indices, sel_idx in outer_bins_sel_idx:
            outer_axes_str=f"{outer_axes} bin {bin_indices}"
            print(f"checking {outer_axes_str} if {axis} bins are ok  with total: {len(sel_idx)}")

            ok = self._optimise_bins_loop_work(outer_axes_str, axis, sel_idx)
    ###

    ###
    def _compute_sel_idx_axis_score(self, axis, sel_idx):
        # how far the sel_idx events inside the axis range are above the thresholds for axis

        # index the numpy arrays directly: building a DataFrame with iloc for every outer bin is slow
        _, mc_counts, data_counts = SampleBinOptimiser.get_bins_and_counts(axis,
                                                self.df[axis].to_numpy()[sel_idx],
                                                self.df['EventWeight'].to_numpy()[sel_idx],
                                                self.optimised_bins[axis],
                                                self.exposure_scaling
                                                )
        # exclude the last bin overflow: inf
        mc = mc_counts[:-1].sum()
        data = data_counts[:-1].sum()
        min_data = self.min_data * self.axes_factors[axis]
        return min(mc / self.min_mc if self.min_mc > 0 else np.inf,
                   data / min_data if min_data > 0 else np.inf)
    ###

    ###
    def _optimise_axis_joint(self, outer_n_bins, outer_axes, axis):
        # joint mode: optimise axis' edges using all the outer bins at once
        edges = self.optimised_bins[axis]
        n_bins = len(edges) - 2
        event_bin_indices = self.df[axis+'binIndex'].to_numpy()

        outer_shape = list(outer_n_bins)
        n_outer_bins = int(np.prod(outer_shape))

        # exclude events assigned to underflow and overflow bins for axis being optimised
        sel = (event_bin_indices >= 1) & (event_bin_indices <= n_bins)

        # build outer bin flat indices and corresponding mask
        outer_bin_flat_indices = np.zeros(len(self.df), dtype=np.int64)
        for o_a, n_o_a_bins in zip(outer_axes, outer_shape):
            event_o_a_indices = self.df[o_a+'binIndex'].to_numpy()
            outer_bin_flat_indices = outer_bin_flat_indices * n_o_a_bins + (event_o_a_indices - 1)

            # exclude events assigned to underflow and overflow bins for outer axis(es)
            sel &= (event_o_a_indices >= 1) & (event_o_a_indices <= n_o_a_bins)

        # events of every (flat_outer_bin, starting bins of {axis} being optimised) 
        sel_bins_flat_indices = outer_bin_flat_indices[sel] * n_bins + (event_bin_indices[sel] - 1)

        # compute mc and data counts in one pass over the events
        mc_counts = np.bincount(sel_bins_flat_indices, minlength=n_outer_bins*n_bins).reshape(n_outer_bins, n_bins)
        data_counts = np.bincount(sel_bins_flat_indices, weights=self.df['EventWeight'].to_numpy()[sel],
                           minlength=n_outer_bins*n_bins).reshape(n_outer_bins, n_bins) * self.exposure_scaling

        min_data = self.min_data * self.axes_factors[axis]
        # TODO add ability to modify required MC counts as well
    
        new_axis_bin_indices = []
        weakest_oa_bin_counts = []
        current_bin_start_idx = 0
        # arrays of the sums (across outer bins) for the {axis} bin being optimised
        mc_count_sums = np.zeros(n_outer_bins)
        data_count_sums = np.zeros(n_outer_bins)
        for bin_idx in range(n_bins):
            mc_count_sums += mc_counts[:, bin_idx]
            data_count_sums += data_counts[:, bin_idx]
            if (mc_count_sums >= self.min_mc).all() and (data_count_sums >= min_data).all():
                new_axis_bin_indices.append(current_bin_start_idx)
                # outer bin with the fewest data (expected) events, and its data and MC counts
                weakest_oa_bin_idx = int(np.argmin(data_count_sums))
                weakest_oa_bin_counts.append((weakest_oa_bin_idx, data_count_sums[weakest_oa_bin_idx],
                                              mc_count_sums[weakest_oa_bin_idx]))
                current_bin_start_idx = bin_idx + 1
                mc_count_sums[:] = 0.
                data_count_sums[:] = 0.

        if not new_axis_bin_indices:
            mc_sum, data_sum = mc_counts.sum(axis=1), data_counts.sum(axis=1)
            bin_idx = int(np.flatnonzero((mc_sum < self.min_mc) | (data_sum < min_data))[0])
            raise RuntimeError(f"all bins are bad! data sum {data_sum[bin_idx]} mc sum {mc_sum[bin_idx]}")
        if len(new_axis_bin_indices) == 1 and n_bins > 1:
            raise RuntimeError(f"[ERROR] axis {axis} fully merged!")

        # the lower edge of each new bin, then the upper edge of the range and inf
        new_edges = np.append(edges[new_axis_bin_indices], edges[-2:])

        bins_changed = len(new_edges) != len(edges)
        if bins_changed:
            print(f"joint: {axis} over {n_outer_bins} outer bins of {outer_axes}: {n_bins} -> {len(new_axis_bin_indices)} bins")
        else:
            print(f"joint: {axis} over {n_outer_bins} outer bins of {outer_axes}: all {n_bins} bins pass, unchanged")
        # per-bin details when the axis is optimised across outer axes, or when it is unchanged
        if outer_axes or not bins_changed:
            for edge_idx, (weakest_outer_bin_index, weakest_data_count, weakest_mc_count) in enumerate(weakest_oa_bin_counts):
                # lower and upper edge of the new {axis} bin
                bin_str = f"  bin [{new_edges[edge_idx]:.4g}, {new_edges[edge_idx+1]:.4g}):"
                if outer_axes:
                    where = [int(i)+1 for i in np.unravel_index(weakest_outer_bin_index, outer_shape)]
                    print(f"{bin_str} weakest outer bin by data is {outer_axes} {where}, data {weakest_data_count:.1f}, mc {weakest_mc_count:.0f}")
                else:
                    print(f"{bin_str} data {weakest_data_count:.1f}, mc {weakest_mc_count:.0f}")

        self.optimised_bins[axis] = new_edges
        self.df[axis+'binIndex'] = np.digitize(self.df[axis].to_numpy(), new_edges, right=False)
    ###

    ###
    def _optimise_bins_loop_work(self, outer_axes_str, axis, idx_sel):
    
        while True:

            df_sel=self.df.iloc[idx_sel]
            data=df_sel[axis].to_numpy()
            weights=df_sel['EventWeight'].to_numpy()

            ok, indices, new_edges = SampleBinOptimiser.optimise_edges(axis,
                                                    data,
                                                    weights, 
                                                    self.optimised_bins[axis],
                                                    self.exposure_scaling,
                                                    self.min_mc,
                                                    self.min_data*self.axes_factors[axis]
                                                    )
            if ok:
                print(f"{outer_axes_str} {axis} ok")
                #self.df.iloc[idx_sel, self.df.columns.get_loc(axis+'binIndex')]=indices
                break
            else:
                print(f"{outer_axes_str} {axis} were optimised")
                self.optimised_bins[axis]=new_edges
                self.df[axis + 'binIndex'] = np.digitize(self.df[axis].to_numpy(),
                                                        self.optimised_bins[axis], right=False)

        return True
    ###

    ###
    @staticmethod
    def optimise_edges(axis, data, weights, edges, data_scaling, min_mc, min_data):

        new_edges = edges.copy()

        indices, mc_counts, data_counts = SampleBinOptimiser.get_bins_and_counts(axis, data, weights, edges, data_scaling)

        n_bins=len(mc_counts[:-1])

        ok=False
        bad_edges=[]
        bin_idx = 0
        while bin_idx < n_bins:
            mc_count=mc_counts[bin_idx]
            data_count=data_counts[bin_idx]
            while True:
                if mc_count < min_mc or data_count < min_data:
                    #print(f"bad bin: {bin_idx} {mc_count}, {data_count}")

                    # last bin, special
                    if bin_idx == n_bins-1:
                        # actually need to merge with last good bin
                        if len(new_edges) < 4:
                            raise RuntimeError(f"all bins are bad! data sum {np.sum(data_counts[:-1])} mc sum {np.sum(mc_counts[:-1])}")
                        else:
                            bad_edge=new_edges[::-1][2]
                            new_edges=np.delete(new_edges, np.where(new_edges == bad_edge))
                            bad_edges.append(bad_edge)
                            bin_idx += 1
                            break
                    else:
                        bin_idx += 1
                        bad_edge=edges[bin_idx]
                        new_edges=np.delete(new_edges, np.where(new_edges == bad_edge))
                        bad_edges.append(bad_edge)
                        mc_count=mc_count+mc_counts[bin_idx]
                        data_count=data_count+data_counts[bin_idx]
                        #print(f"merged {mc_count}, {data_count}")
                else:
                    bin_idx += 1
                    break

        data_sum=np.sum(data_counts[:-1])
        mc_sum=np.sum(mc_counts[:-1])
        #print(f"edges: {edges[:-1]}")
        #print(f"data counts: {data_counts[:-1]}, data sum: {data_sum}")
        #print(f"mc counts: {mc_counts[:-1]}, mc sum:  {mc_sum}")

        if (len(new_edges)==len(edges)):
            ok=True
        else:
            #print(f"removed edges: {bad_edges}")
            #print(f"new {axis} edges: {new_edges}")
            if (len(new_edges)==3):
                #print(f"[WARNING] axis {axis} fully merged! data sum {data_sum} mc sum {mc_sum}")
                raise RuntimeError(f"[ERROR] axis {axis} fully merged!")

        return ok, indices, new_edges
    ###

    ###
    @staticmethod
    def get_bins_and_counts(axis, data, weights, edges, data_scaling):
        indices = np.digitize(data, edges, right=False)

        n = len(edges)
        mc_counts = np.bincount(indices, minlength=n+1)[1:n]
        data_counts = np.bincount(indices, weights=weights, minlength=n+1)[1:n] * data_scaling
        return indices, mc_counts, data_counts
    ###

optimised_sample_bins=dict.fromkeys(samples.keys())

# order of axes here determines optimisation order
histo_axes_bins_initial = {
                        'RecoENu': E_nu_bins,
                        'RecoY': np.arange(0., 1.01, 0.05),
                        'RecoCZ': np.arange(-1., 1.01, 0.2),
                        }
exposure=400.

for sample, sample_id in samples.items():
    
    sample_bin_opt=SampleBinOptimiser(sample, 
                                      df_events[df_events['Sample']==sample_id], 
                                      histo_axes_bins_initial,
                                      {'RecoCZ': 1., 'RecoENu': 120., 'RecoY':15. , },
                                      exposure=exposure
                                      )
    print(f"{sample}")
    optimised_sample_bins[sample]=sample_bin_opt.optimise_bins()
    print("")

print(f"exposure {exposure} kt-y")

for sample in samples.keys():
    for axis, edges in optimised_sample_bins[sample].items():
        print(f"Final {sample} {axis} edges: {edges[:-1]}")
        print("")